# Q3 — Brain-wide visual-response latency and recruitment order

**Lead:** Arash Kanafchian · **Presentation:** slides 6–13

This is the single submission notebook for Question 3. It preserves the complete
unit-level and population-trajectory analyses from the final project notebook,
while making paths, artifacts, and authorship explicit and portable.


### Contribution — Arash Kanafchian

- **Role:** Question framing, latency analysis, anatomical grouping, validation, and reporting
- **Presentation slides:** 6–13
- **Code provenance:** All Q3-specific analytical code in this notebook is attributed to Arash; data-access utilities retain their upstream IBL/Neuromatch provenance.

The attribution in this section applies until the next contribution heading.


In [ ]:
# Shared, portable project setup
from pathlib import Path
import sys

_start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / 'README.md').exists() and (p / 'src').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ibl_q3.artifacts import ArtifactStore
from ibl_q3.config import get_execution_mode, get_paths

PATHS = get_paths(PROJECT_ROOT)
EXECUTION_MODE = get_execution_mode()
print(f'Execution mode: {EXECUTION_MODE.value}')
print(f'Data root: {PATHS.data_root}')
print(f'Artifact root: {PATHS.artifact_root}')

Q3_OUTPUT_DIR = PATHS.namespace('q3')
Q3_STORE = ArtifactStore(Q3_OUTPUT_DIR)

# Q3 — Latency of visual responses and their propagation across the brain

**Q3 (verbatim).** *For responsive units compute the latency after stimulus onset of the
response. Group the regions anatomically. Do the latencies make sense in terms of how
sensory signals propagate through the brain?*

### Two notions of "response", answering two different questions

| | Definition of the response signal | Question it answers | Role here |
|---|---|---|---|
| **Population trajectory** | left-vs-right population distance `d(t)`; latency = first crossing of 70 % of `max d − min d` | *When does a region begin to carry information about **which side** the stimulus appeared on?* | **the reported Q3 latency** — same definition as the reference paper |
| **Unit level** | `abs(PSTH(t) − baseline)` for each neuron | *When does an individual neuron change its firing relative to before the stimulus?* | companion analysis: which units respond, and when |

Both are reported. They are **not** interchangeable, and only the population number is ever
compared with the paper. The unit-level analysis is what makes the answer to Q3 literal
("for responsive units…"); the population analysis is what makes it comparable.

### Reference

International Brain Laboratory et al., *A brain-wide map of neural activity during complex
behaviour*, **Nature 645** (2025), `PaperForQ3.pdf`. Methods, *Population trajectory
analysis*: latency is the first time `d(t) = min d + 0.7 (max d − min d)`; regional
significance comes from 1,000 pseudo-trial permutations at `P < 0.01`.

The paper reports five regional latencies explicitly:
`LGd < VISp ≈ LP < VISpm < VISam`, at **34, 42, 42, 57 and 78 ms**, followed by a later wave
in MRN, SCm, PRNr, IRN and GRN at about **100–120 ms**.

### Resolution caveat (carried through to the comparison)

The paper re-binned raw spikes in **12.5 ms windows at a 2 ms stride**. The Neuromatch files
distributed for this project contain **precomputed 10 ms PSTHs**, giving only 16 samples
across the 0–150 ms window. The floor on our temporal resolution is therefore about ±5 ms,
which is the size of most of the residual disagreement with the paper. This is quantified in
section 6 rather than tuned away.

## 1. Setup, parameters and data loading

Environment, the Neuromatch data helpers, the anatomical (Beryl → Cosmos) region mapping, and
the parameters shared by every later section.

In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    # find_spec raises (rather than returning None) when the parent
    # package is absent, which is the normal case outside Colab.
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'ONE-api', 'ibllib', 'pyarrow'
    ])
    print('Colab dependencies installed.')
else:
    print('Local runtime detected; using the current Python environment.')



import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = PATHS.root
OUTPUT_DIR = Q3_OUTPUT_DIR

# Reuse an existing local Neuromatch cache before downloading anything.
if IN_COLAB:
    DATA_CACHE_DIR = PROJECT_ROOT / 'one_cache'
else:
    cache_candidates = [
        PATHS.data_root,
        Path.home() / 'Downloads/ONE/openalyx.internationalbrainlab.org',
        PATHS.data_root,
    ]
    discovered = next(
        Path.home().glob('Downloads/ONE/**/Neuromatch/data_stimOn'),
        None,
    )
    if discovered is not None:
        cache_candidates.insert(0, discovered.parent.parent)
    DATA_CACHE_DIR = next(
        (path for path in cache_candidates
         if (path / 'Neuromatch/data_stimOn').exists()),
        PATHS.data_root,
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm
from IPython.display import display
from scipy import sparse

from iblatlas.atlas import BrainRegions
from iblutil.util import Bunch
from one.api import ONE
from one.remote.aws import s3_download_file

os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international', cache_dir=DATA_CACHE_DIR)

local_data = DATA_CACHE_DIR / 'Neuromatch/data_stimOn'
if local_data.exists():
    print(f'Reusing existing data: {local_data.resolve()}')
else:
    print(f'No local stimOn data found; it will be downloaded to: {DATA_CACHE_DIR.resolve()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')

In [ ]:
def download_data(event):
    if event not in ['firstMove', 'stimOn', 'feedback']:
        raise ValueError("event must be 'firstMove', 'stimOn', or 'feedback'")
    fname = f'data_{event}.zip'
    remote_path = f'sample_data/Neuromatch/{fname}'
    save_path = one.cache_dir.joinpath('Neuromatch', fname)
    save_path.parent.mkdir(exist_ok=True, parents=True)
    downloaded_file = s3_download_file(remote_path, save_path)
    with zipfile.ZipFile(downloaded_file, 'r') as zip_ref:
        zip_ref.extractall(save_path.parent)


def get_data_path(event):
    return one.cache_dir.joinpath('Neuromatch', f'data_{event}')


def load_metadata(event):
    data_path = get_data_path(event)
    metadata = Bunch()
    metadata['clusters'] = pd.read_parquet(data_path.joinpath('clusters.pqt'))
    metadata['trials'] = pd.read_parquet(data_path.joinpath('trials.pqt'))
    metadata['sessions'] = pd.read_parquet(data_path.joinpath('sessions.pqt'))
    metadata['times'] = np.load(data_path.joinpath('t.npy'))
    metadata['nbins'] = metadata['times'].size
    metadata['dt'] = np.round(np.median(np.diff(metadata['times'])), 2)
    metadata['data_path'] = data_path
    return metadata


def load_psth(data_path, pid, nbins=150):
    psth = sparse.load_npz(data_path.joinpath(f'{pid}.npz')).toarray()
    return psth.reshape(psth.shape[0], -1, nbins)


def get_psth_for_insertion(pid, meta, reg=None, uuids=None):
    clusters = meta.clusters[meta.clusters['pid'] == pid]
    clusters = clusters[['acronym', 'pid', 'uuids', 'cluster_id', 'psth_index']]
    psth = load_psth(meta.data_path, pid, nbins=meta.nbins)
    if reg is not None:
        in_region = clusters['acronym'] == reg
        psth = psth[:, in_region.values, :]
        clusters = clusters[in_region].reset_index(drop=True)
    if uuids is not None:
        in_uuid = clusters['uuids'].isin(uuids)
        psth = psth[:, in_uuid.values, :]
        clusters = clusters[in_uuid].reset_index(drop=True)
    eid = meta.sessions[meta.sessions['pid'] == pid].iloc[0]['eid']
    trials = meta.trials[meta.trials['eid'] == eid].reset_index(drop=True)
    return psth / meta['dt'], clusters, trials


if not get_data_path('stimOn').exists():
    download_data('stimOn')

q3_meta = load_metadata('stimOn')
print(f"Insertions: {q3_meta.clusters['pid'].nunique():,}")
print(f"Units: {len(q3_meta.clusters):,}")
print(f"Beryl acronyms in metadata: {q3_meta.clusters['acronym'].nunique():,}")

In [ ]:
Q3_INVALID_ACRONYMS = {'root', 'void', 'unknown', 'unmapped', 'nan', 'None', ''}
Q3_COSMOS_ACRONYM_TO_LABEL = {
    'Isocortex': 'Isocortex', 'OLF': 'Olfactory areas',
    'HPF': 'Hippocampal formation', 'CTXsp': 'Cortical subplate',
    'CNU': 'Cerebral nuclei', 'TH': 'Thalamus', 'HY': 'Hypothalamus',
    'MB': 'Midbrain', 'HB': 'Hindbrain', 'CB': 'Cerebellum'
}
Q3_COSMOS_LABELS = list(Q3_COSMOS_ACRONYM_TO_LABEL.values())
q3_br = BrainRegions()


def q3_cosmos_group(acronym):
    try:
        atlas_ids = np.asarray(q3_br.acronym2id(str(acronym))).ravel()
        if atlas_ids.size == 0:
            return 'Unmapped'
        cosmos = np.asarray(q3_br.id2acronym(atlas_ids[:1], mapping='Cosmos')).ravel()
        if cosmos.size == 0:
            return 'Unmapped'
        return Q3_COSMOS_ACRONYM_TO_LABEL.get(str(cosmos[0]), 'Unmapped')
    except Exception:
        return 'Unmapped'


q3_all_acronyms = sorted({
    str(region) for region in q3_meta.clusters['acronym'].dropna().unique()
    if str(region) not in Q3_INVALID_ACRONYMS
})
q3_region_group = {region: q3_cosmos_group(region) for region in q3_all_acronyms}
q3_anatomical_regions = {
    group: [region for region in q3_all_acronyms if q3_region_group[region] == group]
    for group in Q3_COSMOS_LABELS
}
q3_paper_region_to_group = {
    region: group
    for group, regions in q3_anatomical_regions.items()
    for region in regions
}

q3_mapping_table = pd.DataFrame({
    'acronym': q3_all_acronyms,
    'anatomical_group': [q3_region_group[r] for r in q3_all_acronyms]
})
display(q3_mapping_table.groupby('anatomical_group', observed=True).size().rename('n_regions').to_frame())
print(f"Valid regions retained: {len(q3_paper_region_to_group):,}")

In [ ]:
from ibl_q3.statistics import benjamini_hochberg as q3_bh_fdr

Q3_PAPER_SHUFFLES = 1000
Q3_PAPER_MIN_UNITS = 1
Q3_PAPER_RELIABLE_MIN_UNITS = 20
Q3_PAPER_FDR_ALPHA = 0.01
Q3_PAPER_POINT_ALPHA = 0.01
Q3_BOOTSTRAP_REPLICATES = 300
Q3_BOOTSTRAP_CI = 0.90
Q3_BOOTSTRAP_SEED = 20260823
Q3_PAPER_SEED = 20260820
Q3_ONSET_FRACTION = 0.70
Q3_PAPER_LATENCY_STRIDE_S = 0.002


def q3_paper_shuffled_labels(labels, strata, n_shuffles, rng):
    """Shuffle labels within each stratum while preserving label counts."""

    pseudo = np.zeros((n_shuffles, len(labels)), dtype=np.float32)
    rows = np.arange(n_shuffles)[:, None]

    for stratum in np.unique(strata):
        trial_index = np.flatnonzero(strata == stratum)
        n_left = int(labels[trial_index].sum())

        if n_left == 0:
            continue
        if n_left == len(trial_index):
            pseudo[:, trial_index] = 1
            continue

        scores = rng.random((n_shuffles, len(trial_index)))
        selected = np.argpartition(
            scores, n_left - 1, axis=1
        )[:, :n_left]
        pseudo[rows, trial_index[selected]] = 1

    return pseudo


def q3_paper_latency(
    times, distance, point_p=None, fraction=Q3_ONSET_FRACTION,
    require_significance=False, interpolate=True
):
    """First crossing of `fraction` of the modulation amplitude, in ms.

    `require_significance=False` (the default) is the paper's rule exactly: the
    first time the distance curve reaches min + fraction * (max - min).
    `require_significance=True` additionally demands the pointwise pseudo-trial
    test at that sample. That gate is stricter than the paper and can only move
    the estimate later, so it is reported alongside as a robustness check and
    never as the headline number.

    Linear interpolation is applied only when the two adjacent samples genuinely
    bracket the threshold. Otherwise the sampled bin is used, and the result is
    always clamped to the analysis window - extrapolating outside it produced the
    impossible pre-stimulus and post-window estimates seen in earlier runs.
    """

    threshold = distance.min() + fraction * np.ptp(distance)
    above = distance >= threshold
    if require_significance:
        above = above & (point_p < Q3_PAPER_POINT_ALPHA)
    eligible = np.flatnonzero(above)
    if len(eligible) == 0:
        return np.nan

    stop = int(eligible[0])
    brackets = (
        interpolate
        and stop > 0
        and distance[stop - 1] < threshold <= distance[stop]
    )
    if brackets:
        crossing = times[stop - 1] + (
            (threshold - distance[stop - 1])
            / (distance[stop] - distance[stop - 1])
            * (times[stop] - times[stop - 1])
        )
    else:
        crossing = times[stop]

    crossing = float(np.clip(crossing, times[0], times[-1]))
    return float(
        np.round(crossing / Q3_PAPER_LATENCY_STRIDE_S)
        * Q3_PAPER_LATENCY_STRIDE_S * 1000
    )


# Reported interval: 95%, at the insertion (recording) level - see section 5.
Q3_INSERTION_BOOTSTRAP_REPLICATES = 1000
Q3_REPORTED_CI = 0.95

## 2. Responsive-unit identification

A neuron is tested by comparing its firing in a **baseline window of −200 to 0 ms** with the
**response window of 50 to 150 ms**, trial by trial.

| Criterion | Setting |
|---|---|
| Statistical test | Mann–Whitney U (two-sided), trials as samples, vectorised across units |
| Multiple-comparison correction | Benjamini–Hochberg FDR across **all** tested units, brain-wide |
| Significance threshold | `q < 0.01` |
| Minimum effect size | `abs(mean response − mean baseline) ≥ 0.20 Hz` |
| Onset requirement | a finite onset latency must exist inside 0–150 ms |

**Exact Boolean definition of a responsive unit** (this is the only place units are selected):

```python
responsive = (q_value < 0.01) & (effect_hz >= 0.20) & latency_ms.notna()
```

Onset for a responsive unit is the **first sustained crossing** of 70 % of its peak
baseline-subtracted response, where "sustained" means the threshold is exceeded in that bin
**and** the next one (2 × 10 ms bins). Regions excluded from the population analysis
(`root`, `void`, unmapped) are excluded here too, so the two paths describe the same brain.

In [ ]:
from scipy import stats
from scipy.ndimage import gaussian_filter1d

Q3_BASELINE_WINDOW = (-0.200, 0.000)
Q3_RESPONSE_WINDOW = (0.050, 0.150)
Q3_LATENCY_WINDOW = (0.000, 0.150)
Q3_UNIT_FDR_ALPHA = 0.01
Q3_MIN_EFFECT_HZ = 0.20
Q3_SUSTAINED_BINS = 2  # 20 ms using the tutorial's 10-ms PSTHs
Q3_MIN_RESPONSIVE_UNITS = 5
Q3_MIN_PIDS = 2


def q3_first_sustained_crossing(is_above, n_bins=2):
    # Start from the crossing itself: initialising to all-True and AND-ing only
    # with shifted copies returned the bin *before* the crossing and never
    # enforced the sustained requirement at all.
    sustained = is_above.copy()
    for shift in range(1, n_bins):
        sustained[:, :-shift] &= is_above[:, shift:]
        sustained[:, -shift:] = False
    found = sustained.any(axis=1)
    first = np.argmax(sustained, axis=1).astype(float)
    first[~found] = np.nan
    return first

In [ ]:
baseline_bins = (
    (q3_meta.times >= Q3_BASELINE_WINDOW[0])
    & (q3_meta.times < Q3_BASELINE_WINDOW[1])
)
response_bins = (
    (q3_meta.times >= Q3_RESPONSE_WINDOW[0])
    & (q3_meta.times < Q3_RESPONSE_WINDOW[1])
)
latency_bins = (
    (q3_meta.times >= Q3_LATENCY_WINDOW[0])
    & (q3_meta.times <= Q3_LATENCY_WINDOW[1])
)
latency_times = q3_meta.times[latency_bins]


q3_unit_rows = []
for pid in tqdm.tqdm(
    q3_meta.sessions['pid'].dropna().unique(),
    desc='Q3 responsive units'
):
    psth_pid, clusters_pid, trials_pid = get_psth_for_insertion(pid, q3_meta)
    if psth_pid is None or len(trials_pid) == 0:
        continue

    # Same region-validity filter as the population analysis, so 'root' and
    # other unmapped labels cannot reach the unit-level exports.
    valid_units = ~clusters_pid['acronym'].astype(str).isin(Q3_INVALID_ACRONYMS)
    if not valid_units.any():
        continue
    psth_pid = psth_pid[:, valid_units.to_numpy(), :]
    clusters_pid = clusters_pid.loc[valid_units].reset_index(drop=True)

    baseline_rate = psth_pid[:, :, baseline_bins].mean(axis=2)
    response_rate = psth_pid[:, :, response_bins].mean(axis=2)

    # Paper-adjacent overall responsiveness test: baseline versus 50-150 ms.
    # The test is vectorized across units.
    _, p_value = stats.mannwhitneyu(
        baseline_rate,
        response_rate,
        axis=0,
        alternative='two-sided',
        method='asymptotic'
    )
    effect_hz = np.abs(
        np.nanmean(response_rate, axis=0)
        - np.nanmean(baseline_rate, axis=0)
    )

    mean_trace = gaussian_filter1d(
        np.nanmean(psth_pid, axis=0), sigma=1, axis=1
    )
    baseline_mean = mean_trace[:, baseline_bins].mean(axis=1)
    evoked = np.abs(
        mean_trace[:, latency_bins] - baseline_mean[:, None]
    )
    peak = np.nanmax(evoked, axis=1)
    threshold = Q3_ONSET_FRACTION * peak
    onset_bin = q3_first_sustained_crossing(
        evoked >= threshold[:, None], Q3_SUSTAINED_BINS
    )
    latency_ms = np.full(len(clusters_pid), np.nan)
    has_latency = np.isfinite(onset_bin) & (peak > 0)
    latency_ms[has_latency] = (
        latency_times[onset_bin[has_latency].astype(int)] * 1000
    )

    result = clusters_pid.copy()
    result['p_value'] = p_value
    result['effect_hz'] = effect_hz
    result['latency_ms'] = latency_ms
    q3_unit_rows.append(result)


q3_all_tested_units = pd.concat(q3_unit_rows, ignore_index=True)
q3_all_tested_units['q_value'] = q3_bh_fdr(
    q3_all_tested_units['p_value'].to_numpy()
)

# THIS IS THE EXACT RESPONSIVE-UNIT DEFINITION.
q3_responsive_mask = (
    (q3_all_tested_units['q_value'] < Q3_UNIT_FDR_ALPHA)
    & (q3_all_tested_units['effect_hz'] >= Q3_MIN_EFFECT_HZ)
    & q3_all_tested_units['latency_ms'].notna()
)
q3_responsive_units = q3_all_tested_units.loc[
    q3_responsive_mask
].copy()

In [ ]:
# Responsive-unit counts alongside the number of units actually tested, so the
# percentage responsive is well defined for every region.
q3_unit_totals = (
    q3_all_tested_units.groupby('acronym', as_index=False)
    .agg(n_tested_units=('uuids', 'nunique'))
)
q3_unit_responsive = (
    q3_responsive_units.groupby('acronym', as_index=False)
    .agg(n_responsive_units=('uuids', 'nunique'))
)
q3_unit_counts = q3_unit_totals.merge(
    q3_unit_responsive, on='acronym', how='left'
).fillna({'n_responsive_units': 0})
q3_unit_counts['percent_responsive'] = (
    100 * q3_unit_counts['n_responsive_units'] / q3_unit_counts['n_tested_units']
)

assert not q3_responsive_units['acronym'].astype(str).isin(
    Q3_INVALID_ACRONYMS
).any(), 'invalid acronym reached the unit-level path'

print(f'Units tested: {len(q3_all_tested_units):,}')
print(f'Responsive units: {len(q3_responsive_units):,} '
      f'({100 * len(q3_responsive_units) / len(q3_all_tested_units):.1f}%)')
print(f'Regions with at least one responsive unit: '
      f"{q3_responsive_units['acronym'].nunique():,}")
display(q3_unit_counts.sort_values('n_responsive_units', ascending=False).head(10).round(1))

## 3. Unit and regional latency estimation

**Per-unit latency** is summarised per region by first taking the median within each
insertion, then the median across insertions. This prevents one large recording from
dominating a region, because neurons within a recording are not independent replicates.

**Regional latency** — the number reported for Q3 — follows the paper's population-trajectory
method: all eligible units (no preselection of responsive units), left-versus-right
trajectories, stimulus labels shuffled only within `block × choice`, 1,000 pseudo-trials, and
BH-FDR at `q < 0.01` for regional significance.

In [ ]:
# First summarize within each PID-region so large recordings do not dominate.
q3_pid_region_latency = (
    q3_responsive_units
    .groupby(['acronym', 'pid'], as_index=False)
    .agg(
        n_responsive_units=('uuids', 'nunique'),
        latency_ms=('latency_ms', 'median')
    )
)

q3_region_latency = (
    q3_pid_region_latency
    .groupby('acronym', as_index=False)
    .agg(
        n_pids=('pid', 'nunique'),
        n_responsive_units=('n_responsive_units', 'sum'),
        latency_ms=('latency_ms', 'median'),
        q25_ms=('latency_ms', lambda x: x.quantile(0.25)),
        q75_ms=('latency_ms', lambda x: x.quantile(0.75)),
    )
)
q3_region_latency['passes_coverage'] = (
    (q3_region_latency['n_pids'] >= Q3_MIN_PIDS)
    & (q3_region_latency['n_responsive_units'] >= Q3_MIN_RESPONSIVE_UNITS)
)


# Reuse the previously defined Cosmos mapper; no region list is hard-coded.
q3_region_latency['anatomical_group'] = (
    q3_region_latency['acronym'].map(q3_cosmos_group)
)
Q3_COSMOS_ORDER = [
    'Isocortex', 'Olfactory areas', 'Hippocampal formation',
    'Cortical subplate', 'Cerebral nuclei', 'Thalamus',
    'Hypothalamus', 'Midbrain', 'Hindbrain', 'Cerebellum'
]
q3_region_latency['anatomical_group'] = pd.Categorical(
    q3_region_latency['anatomical_group'],
    categories=Q3_COSMOS_ORDER,
    ordered=True
)
q3_region_latency = q3_region_latency.sort_values(
    ['anatomical_group', 'latency_ms', 'acronym']
).reset_index(drop=True)

In [ ]:
q3_paper_region_to_group = {
    region: group
    for group, regions in q3_anatomical_regions.items()
    for region in regions
}
q3_paper_window = (
    (q3_meta.times >= 0) & (q3_meta.times <= 0.15)
)
q3_paper_times = q3_meta.times[q3_paper_window]
q3_paper_rng = np.random.default_rng(Q3_PAPER_SEED)

# Accumulate squared cell-wise differences. This gives exactly the same
# regional Euclidean distance as concatenating all cells, without storing
# every PID in memory at once.
q3_true_sumsq = {
    region: np.zeros(q3_paper_window.sum())
    for region in q3_paper_region_to_group
}
q3_null_sumsq = {
    region: np.zeros((Q3_PAPER_SHUFFLES, q3_paper_window.sum()))
    for region in q3_paper_region_to_group
}
q3_paper_n_units = dict.fromkeys(q3_paper_region_to_group, 0)
q3_paper_pids = {
    region: set() for region in q3_paper_region_to_group
}
q3_paper_units_per_session = {
    region: {} for region in q3_paper_region_to_group
}

# Retain per-PID true-distance contributions for the optional matched-PID
# raw-resolution pilot; this does not alter the main calculation.
q3_true_sumsq_by_pid = {}
q3_n_units_by_pid = {}

Q3_TRIAL_BOOTSTRAP_REGIONS = {
    'LGd', 'LP', 'LGv', 'NOT', 'SCs', 'OP', 'APN', 'VISp',
    'VISl', 'VISam', 'VISpm', 'VISal', 'AUDv', 'PRNc', 'MRN',
    'PRNr', 'GRN', 'SCm', 'IRN', 'PARN', 'RN', 'ACAd', 'CP',
    'ACB', 'MOp', 'MOs'
}
q3_trial_boot_rng = np.random.default_rng(Q3_BOOTSTRAP_SEED)
q3_trial_boot_sumsq = {
    region: np.zeros((Q3_BOOTSTRAP_REPLICATES, q3_paper_window.sum()))
    for region in Q3_TRIAL_BOOTSTRAP_REGIONS
    if region in q3_paper_region_to_group
}

q3_paper_target_pids = q3_meta.clusters.loc[
    q3_meta.clusters['acronym'].isin(q3_paper_region_to_group), 'pid'
].dropna().unique()

In [ ]:
for pid in tqdm.tqdm(
    q3_paper_target_pids, desc='Paper-aligned Q3 trajectory'
):
    psth_pid, clusters_pid, trials_pid = get_psth_for_insertion(
        pid, q3_meta
    )
    eid = q3_meta.sessions.loc[
        q3_meta.sessions['pid'] == pid, 'eid'
    ].iloc[0]
    target_units = clusters_pid['acronym'].isin(
        q3_paper_region_to_group
    ).to_numpy()
    if not target_units.any():
        continue

    left_contrast = trials_pid['contrastLeft'].to_numpy(dtype=float)
    right_contrast = trials_pid['contrastRight'].to_numpy(dtype=float)
    choice = trials_pid['choice'].to_numpy(dtype=float)
    block = trials_pid['probabilityLeft'].to_numpy(dtype=float)

    # The paper code uses every assigned stimulus side, including zero
    # contrast; side is represented by which contrast column is non-NaN.
    left_label = np.isfinite(left_contrast)
    right_label = np.isfinite(right_contrast)
    keep = (
        (left_label | right_label) &
        np.isfinite(choice) &
        np.isfinite(block)
    )
    labels = left_label[keep]
    if not labels.any() or labels.all():
        continue

    # Shuffle only within trials sharing the exact block and choice side.
    strata = pd.factorize(pd.MultiIndex.from_arrays([
        block[keep],
        choice[keep],
    ]))[0]
    pseudo = q3_paper_shuffled_labels(
        labels, strata, Q3_PAPER_SHUFFLES, q3_paper_rng
    )
    pseudo_left_n = pseudo.sum(axis=1)
    pseudo_right_n = len(labels) - pseudo_left_n
    if np.any(pseudo_left_n == 0) or np.any(pseudo_right_n == 0):
        continue

    values = psth_pid[keep][:, target_units, :][:, :, q3_paper_window]
    values = values.astype(np.float32)
    target_clusters = clusters_pid.loc[target_units].reset_index(drop=True)

    # Trial bootstrap weights preserve stimulus side and block x choice
    # counts while resampling trials with replacement inside each stratum.
    bootstrap_weights = np.zeros(
        (Q3_BOOTSTRAP_REPLICATES, len(labels)), dtype=np.float32
    )
    for stratum in np.unique(strata):
        for side in (False, True):
            trial_index = np.flatnonzero(
                (strata == stratum) & (labels == side)
            )
            if len(trial_index) == 0:
                continue
            bootstrap_weights[:, trial_index] = (
                q3_trial_boot_rng.multinomial(
                    len(trial_index),
                    np.full(len(trial_index), 1 / len(trial_index)),
                    size=Q3_BOOTSTRAP_REPLICATES
                )
            )

    for region, region_units in target_clusters.groupby('acronym'):
        region_index = region_units.index.to_numpy()
        region_values = values[:, region_index, :]

        true_difference = (
            region_values[labels].mean(axis=0) -
            region_values[~labels].mean(axis=0)
        )
        true_contribution = np.square(true_difference).sum(axis=0)
        q3_true_sumsq[region] += true_contribution
        q3_true_sumsq_by_pid[(region, pid)] = true_contribution
        q3_n_units_by_pid[(region, pid)] = region_values.shape[1]

        if region in q3_trial_boot_sumsq:
            left_flat = region_values[labels].reshape(labels.sum(), -1)
            right_flat = region_values[~labels].reshape((~labels).sum(), -1)
            bootstrap_left = (
                bootstrap_weights[:, labels] @ left_flat
            ) / labels.sum()
            bootstrap_right = (
                bootstrap_weights[:, ~labels] @ right_flat
            ) / (~labels).sum()
            bootstrap_difference = (
                bootstrap_left - bootstrap_right
            ).reshape(
                Q3_BOOTSTRAP_REPLICATES,
                region_values.shape[1], region_values.shape[2]
            )
            q3_trial_boot_sumsq[region] += np.square(
                bootstrap_difference
            ).sum(axis=1)

        flat_values = region_values.reshape(len(labels), -1)
        pseudo_left = (
            pseudo @ flat_values
        ) / pseudo_left_n[:, None]
        pseudo_right = (
            (1 - pseudo) @ flat_values
        ) / pseudo_right_n[:, None]
        pseudo_difference = (pseudo_left - pseudo_right).reshape(
            Q3_PAPER_SHUFFLES,
            region_values.shape[1],
            region_values.shape[2]
        )
        q3_null_sumsq[region] += np.square(
            pseudo_difference
        ).sum(axis=1)

        n_region_units = region_values.shape[1]
        q3_paper_n_units[region] += n_region_units
        q3_paper_pids[region].add(pid)
        session_counts = q3_paper_units_per_session[region]
        session_counts[eid] = session_counts.get(eid, 0) + n_region_units

In [ ]:
q3_paper_rows = []
q3_paper_point_p = {}
# Cache each region's distance curve (266 regions x 16 samples): it makes every
# sensitivity check in section 6 rerun instantly, without touching the loop above.
q3_region_distance = {}
for region in q3_paper_region_to_group:
    n_units = q3_paper_n_units[region]
    session_counts = q3_paper_units_per_session[region]
    n_sessions = len(session_counts)
    n_sessions_ge5 = sum(count >= 5 for count in session_counts.values())
    passes_coverage = (n_units >= 20) and (n_sessions_ge5 >= 2)

    row = {
        'acronym': region,
        'anatomical_group': q3_paper_region_to_group[region],
        'n_pids': len(q3_paper_pids[region]),
        'n_sessions': n_sessions,
        'n_sessions_ge5': n_sessions_ge5,
        'n_units': n_units,
        'amplitude_hz': np.nan,
        'latency_ms': np.nan,
        'latency_gated_ms': np.nan,
        'trial_ci_low_ms': np.nan,
        'trial_ci_high_ms': np.nan,
        'trial_bootstrap_valid': 0,
        'p_value': np.nan,
        'passes_coverage': passes_coverage,
    }
    if n_units == 0:
        q3_paper_rows.append(row)
        continue

    distance = np.sqrt(q3_true_sumsq[region] / n_units)
    null_distance = np.sqrt(q3_null_sumsq[region] / n_units)

    # Match state_space_bwm.py: subtract each curve's own minimum.
    distance_zeroed = distance - distance.min()
    null_zeroed = null_distance - null_distance.min(axis=1, keepdims=True)
    amplitude = distance_zeroed.max()
    null_amplitude = null_zeroed.max(axis=1)
    amplitude_p = (
        1 + np.sum(null_amplitude >= amplitude)
    ) / (Q3_PAPER_SHUFFLES + 1)
    point_p = (
        1 + np.sum(null_zeroed >= distance_zeroed[None, :], axis=0)
    ) / (Q3_PAPER_SHUFFLES + 1)
    q3_paper_point_p[region] = point_p
    q3_region_distance[region] = distance_zeroed

    bootstrap_latency = []
    if region in q3_trial_boot_sumsq and n_units > 0:
        trial_boot_distance = np.sqrt(
            q3_trial_boot_sumsq[region] / n_units
        )
        trial_boot_distance -= trial_boot_distance.min(
            axis=1, keepdims=True
        )
        for sampled_distance in trial_boot_distance:
            bootstrap_latency.append(
                q3_paper_latency(q3_paper_times, sampled_distance)
            )
    bootstrap_latency = np.asarray(bootstrap_latency, dtype=float)
    bootstrap_latency = bootstrap_latency[np.isfinite(bootstrap_latency)]
    ci_low = (1 - Q3_BOOTSTRAP_CI) / 2
    ci_high = 1 - ci_low

    row.update({
        'amplitude_hz': amplitude,
        # Headline number: the paper's plain 70% crossing.
        'latency_ms': q3_paper_latency(q3_paper_times, distance_zeroed),
        # Robustness only: the same crossing, additionally gated on the
        # pointwise pseudo-trial test. Not used for any reported result.
        'latency_gated_ms': q3_paper_latency(
            q3_paper_times, distance_zeroed, point_p,
            require_significance=True
        ),
        'trial_ci_low_ms': (
            np.quantile(bootstrap_latency, ci_low)
            if len(bootstrap_latency) else np.nan
        ),
        'trial_ci_high_ms': (
            np.quantile(bootstrap_latency, ci_high)
            if len(bootstrap_latency) else np.nan
        ),
        'trial_bootstrap_valid': len(bootstrap_latency),
        'p_value': amplitude_p,
    })
    q3_paper_rows.append(row)

In [ ]:
q3_paper_regions = pd.DataFrame(q3_paper_rows)
q3_paper_regions['q_value'] = np.nan
q3_fdr_eligible = q3_paper_regions['passes_coverage']
q3_paper_regions.loc[q3_fdr_eligible, 'q_value'] = q3_bh_fdr(
    q3_paper_regions.loc[q3_fdr_eligible, 'p_value'].to_numpy()
)
q3_paper_regions['significant'] = (
    q3_paper_regions['q_value'] < Q3_PAPER_FDR_ALPHA
)
q3_paper_significant_regions = q3_paper_regions[
    q3_paper_regions['significant'] &
    q3_paper_regions['passes_coverage']
].copy()

q3_paper_group_summary = (
    q3_paper_significant_regions
    .groupby('anatomical_group')
    .agg(
        n_regions=('acronym', 'nunique'),
        n_units=('n_units', 'sum'),
        latency_ms=('latency_ms', 'median'),
        q25_ms=('latency_ms', lambda x: x.quantile(0.25)),
        q75_ms=('latency_ms', lambda x: x.quantile(0.75))
    )
    .sort_values('latency_ms')
)

print(f"All valid regions analysed: {len(q3_paper_regions):,}")
print(f"Regions passing coverage: {int(q3_paper_regions['passes_coverage'].sum()):,}")
print(f"FDR-significant regions that also pass coverage: {len(q3_paper_significant_regions):,}")

In [ ]:
# Insertion-level (PID) bootstrap - the meaningful unit of replication.
#
# Neurons recorded on the same probe insertion share an animal, a session and a
# electrode placement, so they are not independent biological replicates. The
# reported interval therefore resamples *insertions* with replacement and
# recomputes the pooled distance curve from the per-insertion sums already
# accumulated above, so no extra pass over the data is needed.
q3_pid_boot_rng = np.random.default_rng(Q3_BOOTSTRAP_SEED)
q3_insertion_ci = {}
q3_ci_tail = (1 - Q3_REPORTED_CI) / 2

for region in q3_paper_region_to_group:
    pids = sorted(q3_paper_pids[region])
    if len(pids) < 2:
        continue
    sumsq = np.array([q3_true_sumsq_by_pid[(region, pid)] for pid in pids])
    counts = np.array(
        [q3_n_units_by_pid[(region, pid)] for pid in pids], dtype=float
    )
    draws = q3_pid_boot_rng.integers(
        len(pids), size=(Q3_INSERTION_BOOTSTRAP_REPLICATES, len(pids))
    )
    resampled_units = counts[draws].sum(axis=1)
    resampled_sumsq = sumsq[draws].sum(axis=1)
    valid = resampled_units > 0
    resampled_distance = np.sqrt(
        resampled_sumsq[valid] / resampled_units[valid, None]
    )

    latencies = np.array([
        q3_paper_latency(q3_paper_times, curve - curve.min())
        for curve in resampled_distance
    ])
    latencies = latencies[np.isfinite(latencies)]
    if len(latencies) < 0.5 * Q3_INSERTION_BOOTSTRAP_REPLICATES:
        continue
    q3_insertion_ci[region] = (
        float(np.quantile(latencies, q3_ci_tail)),
        float(np.quantile(latencies, 1 - q3_ci_tail)),
        len(latencies),
    )

for column, position in [
    ('latency_ci_low_ms', 0), ('latency_ci_high_ms', 1), ('bootstrap_valid', 2)
]:
    q3_paper_regions[column] = q3_paper_regions['acronym'].map(
        lambda region: q3_insertion_ci.get(region, (np.nan, np.nan, 0))[position]
    )

# Every estimate and every interval must stay inside the analysis window.
# The bin centres run 5-145 ms inside the nominal 0-150 ms window, and rounding
# to the paper's 2 ms stride can move an estimate by up to one stride, so the
# nominal window is the correct bound to assert against.
q3_window_ms = (0.0, 150.0)
q3_edge_ms = (q3_paper_times[0] * 1000, q3_paper_times[-1] * 1000)
for column in ['latency_ms', 'latency_gated_ms', 'latency_ci_low_ms',
               'latency_ci_high_ms', 'trial_ci_low_ms', 'trial_ci_high_ms']:
    finite = q3_paper_regions[column].dropna()
    assert finite.between(*q3_window_ms).all(), (
        f'{column} left the {q3_window_ms} ms analysis window'
    )

q3_ci_ok = q3_paper_regions['latency_ci_low_ms'].notna()
assert (
    q3_paper_regions.loc[q3_ci_ok, 'latency_ci_low_ms']
    <= q3_paper_regions.loc[q3_ci_ok, 'latency_ci_high_ms']
).all(), 'inverted confidence interval'

# Regions whose crossing lands on the first sampled bin are reported, but the
# window cannot resolve them: their true onset may precede the recorded window.
q3_at_first_bin = q3_paper_regions.loc[
    q3_paper_regions['latency_ms'] <= q3_edge_ms[0], 'acronym'
].tolist()

print(f'Insertion-level {Q3_REPORTED_CI:.0%} intervals: '
      f'{len(q3_insertion_ci):,} regions '
      f'({Q3_INSERTION_BOOTSTRAP_REPLICATES:,} replicates each)')
print(f'Latency resolved to the first sampled bin ({q3_edge_ms[0]:.0f} ms), so '
      f'unresolvable within this window: {len(q3_at_first_bin)} regions'
      + (f' -> {", ".join(q3_at_first_bin[:12])}' if q3_at_first_bin else ''))

In [ ]:
q3_paper_regions_complete = q3_paper_regions.sort_values(
    ['anatomical_group', 'latency_ms', 'acronym'], na_position='last'
).reset_index(drop=True)

Q3_REQUIRED_REGIONS = [
    'PRNc', 'GRN', 'ACB', 'MOp', 'PARN', 'MOs', 'RN', 'AUDv', 'VISam'
]
q3_required_audit = (
    q3_paper_regions_complete.set_index('acronym')
    .reindex(Q3_REQUIRED_REGIONS)
    .reset_index()
)
q3_required_audit['present_in_metadata'] = (
    q3_required_audit['n_units'].fillna(0) > 0
)

display(q3_required_audit[[
    'acronym', 'anatomical_group', 'n_sessions_ge5', 'n_units',
    'amplitude_hz', 'latency_ms', 'latency_ci_low_ms',
    'latency_ci_high_ms', 'bootstrap_valid', 'p_value', 'q_value',
    'passes_coverage', 'significant', 'present_in_metadata'
]].round(3))
display(q3_paper_group_summary.round(1))

missing_required = q3_required_audit.loc[
    ~q3_required_audit['present_in_metadata'], 'acronym'
].tolist()
if missing_required:
    raise RuntimeError(f'Required regions missing from dataset: {missing_required}')

## 4. Anatomical propagation and parallel pathways

Regions are placed into anatomical stages **before** looking at their latencies, so the
ordering is a result rather than a construction. Each stage gets an equal-region median: the
line is deliberately *not* weighted by neuron count, because neurons within a region are not
independent regional replicates and a neuron-weighted mean would let the largest regions
decide the shape of the path.

Orange points pass the paper-style global BH-FDR at `q < 0.01`; grey points pass coverage but
not global FDR; red crosses have insufficient coverage. The stage line uses a looser
descriptive screen (raw permutation `p < 0.05`) and does not relabel any point's inferential
status.

The dashed branch is drawn because early visual-midbrain regions (SCs, NOT, OP) respond at
roughly the same time as the visual thalamus rather than after the cortex — a pattern
consistent with parallel retinal output, not with a single serial chain.

In [ ]:
Q3_STAGE_P_ALPHA = 0.05

Q3_PATHWAY_GROUPS = {
    'Visual thalamus': ['LGd', 'LP', 'LGv'],
    'Parallel visual midbrain': ['NOT', 'SCs', 'OP', 'APN'],
    'Primary visual cortex': ['VISp'],
    'Higher / sensory cortex': [
        'VISl', 'VISam', 'VISpm', 'VISal', 'AUDv'
    ],
    'Later midbrain / hindbrain': [
        'PRNc', 'MRN', 'PRNr', 'GRN', 'SCm', 'IRN', 'PARN', 'RN'
    ],
    'Association / action': ['ACAd', 'CP', 'ACB', 'MOp', 'MOs'],
}
Q3_PATHWAY_ORDER = list(Q3_PATHWAY_GROUPS)
region_to_pathway = {
    region: group
    for group, regions in Q3_PATHWAY_GROUPS.items()
    for region in regions
}

q3_pathway = q3_paper_regions_complete[
    q3_paper_regions_complete['acronym'].isin(region_to_pathway)
].copy()
q3_pathway['pathway_group'] = q3_pathway['acronym'].map(region_to_pathway)
q3_pathway['required'] = q3_pathway['acronym'].isin(Q3_REQUIRED_REGIONS)
q3_pathway['status'] = np.select(
    [q3_pathway['significant'], q3_pathway['passes_coverage']],
    ['FDR-significant', 'Not FDR-significant'],
    default='Insufficient coverage'
)
q3_pathway['pathway_group'] = pd.Categorical(
    q3_pathway['pathway_group'], categories=Q3_PATHWAY_ORDER, ordered=True
)

# The solid line uses the paper's explicit canonical ordering. Visual-midbrain
# regions are kept as a parallel branch instead of forcing them between
# thalamus and VISp.
Q3_MAIN_PATH_STAGES = {
    'Visual thalamus': ['LGd', 'LP'],
    'Primary visual cortex': ['VISp'],
    'Higher visual cortex': ['VISpm', 'VISam'],
    'Later midbrain / hindbrain': [
        'PRNc', 'MRN', 'PRNr', 'GRN', 'SCm', 'IRN', 'PARN', 'RN'
    ],
    'Association / action': ['ACAd', 'CP', 'ACB', 'MOp', 'MOs'],
}
Q3_PARALLEL_STAGES = {
    'Early visual-midbrain branch': ['SCs', 'NOT', 'OP'],
    'Late APN branch': ['APN'],
}


def q3_stage_delays(stage_regions):
    rows = []
    for stage, regions in stage_regions.items():
        selected = q3_pathway[
            q3_pathway['acronym'].isin(regions)
            & q3_pathway['passes_coverage']
            & q3_pathway['latency_ms'].notna()
            & (q3_pathway['p_value'] < Q3_STAGE_P_ALPHA)
        ].sort_values('latency_ms')
        rows.append({
            'stage': stage,
            'n_regions': len(selected),
            'n_global_fdr': int(selected['significant'].sum()),
            'regions_used': ', '.join(selected['acronym']),
            'stage_delay_ms': selected['latency_ms'].median(),
            'q25_ms': selected['latency_ms'].quantile(0.25),
            'q75_ms': selected['latency_ms'].quantile(0.75),
        })
    return pd.DataFrame(rows).set_index('stage')


q3_main_stage_summary = q3_stage_delays(Q3_MAIN_PATH_STAGES)
q3_parallel_stage_summary = q3_stage_delays(Q3_PARALLEL_STAGES)

# Exact approximate latencies stated in the paper's main text. Regions only
# described as "about 100-120 ms" are not assigned invented point estimates.
Q3_PAPER_REPORTED_LATENCY_MS = {
    'LGd': 34.0,
    'VISp': 42.0,
    'LP': 42.0,
    'VISpm': 57.0,
    'VISam': 78.0,
}
# Verify that the requested acronyms enter the focused result and plot.
assert set(Q3_REQUIRED_REGIONS).issubset(set(q3_pathway['acronym']))
display(q3_pathway[[
    'acronym', 'pathway_group', 'required', 'n_sessions_ge5', 'n_units',
    'latency_ms', 'latency_ci_low_ms', 'latency_ci_high_ms',
    'bootstrap_valid', 'p_value', 'q_value', 'status'
]].sort_values(['pathway_group', 'latency_ms']).round(3))
print(f'Descriptive stage-line threshold: raw permutation p < {Q3_STAGE_P_ALPHA}')
display(q3_main_stage_summary)
display(q3_parallel_stage_summary)

In [ ]:
STYLE = {
    'FDR-significant': ('#F28E2B', 'o'),
    'Not FDR-significant': ('#B8BEC6', 'o'),
    'Insufficient coverage': ('#C44E52', 'x'),
}

rows, group_centres, y = [], {}, 0
for group in Q3_PATHWAY_ORDER:
    group_rows = q3_pathway[q3_pathway['pathway_group'] == group].sort_values(
        ['latency_ms', 'acronym'], na_position='last'
    )
    start = y
    for _, row in group_rows.iterrows():
        rows.append((y, row))
        y += 1
    group_centres[group] = (start + y - 1) / 2
    y += 0.8

fig, ax = plt.subplots(figsize=(15.5, max(9, 0.37 * len(rows) + 2)))

# One robust delay per main stage, connected in the paper's canonical order.
# Visual midbrain is drawn separately as a candidate parallel branch.
main_y_group = {
    'Visual thalamus': 'Visual thalamus',
    'Primary visual cortex': 'Primary visual cortex',
    'Higher visual cortex': 'Higher / sensory cortex',
    'Later midbrain / hindbrain': 'Later midbrain / hindbrain',
    'Association / action': 'Association / action',
}
stage_plot = q3_main_stage_summary.dropna(subset=['stage_delay_ms'])
stage_y = [group_centres[main_y_group[stage]] for stage in stage_plot.index]
ax.plot(
    stage_plot['stage_delay_ms'], stage_y,
    color='#2457A6', linewidth=1.8, marker='D', markersize=6,
    zorder=2
)
for group, ypos in zip(stage_plot.index, stage_y):
    stage = stage_plot.loc[group]
    ax.errorbar(
        stage['stage_delay_ms'], ypos,
        xerr=np.array([[
            stage['stage_delay_ms'] - stage['q25_ms']
        ], [
            stage['q75_ms'] - stage['stage_delay_ms']
        ]]),
        fmt='none', ecolor='#2457A6', capsize=3,
        linewidth=1.5, zorder=2
    )
    ax.annotate(
        f"{stage['stage_delay_ms']:.0f} ms",
        (stage['stage_delay_ms'], ypos), xytext=(7, 8),
        textcoords='offset points', color='#2457A6', fontsize=8,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=1.2)
    )

region_y = {row['acronym']: ypos for ypos, row in rows}
parallel_plot = q3_parallel_stage_summary.dropna(subset=['stage_delay_ms'])
parallel_y = {
    'Early visual-midbrain branch': np.mean([
        region_y[region] for region in ['SCs', 'NOT', 'OP']
    ]),
    'Late APN branch': region_y['APN'],
}
branch_x = [
    q3_main_stage_summary.loc['Visual thalamus', 'stage_delay_ms'],
    parallel_plot.loc['Early visual-midbrain branch', 'stage_delay_ms'],
    parallel_plot.loc['Late APN branch', 'stage_delay_ms'],
]
branch_y = [
    group_centres['Visual thalamus'],
    parallel_y['Early visual-midbrain branch'],
    parallel_y['Late APN branch'],
]
ax.plot(
    branch_x, branch_y, color='#168C8C', linestyle='--',
    linewidth=1.6, marker='D', markersize=5.5, zorder=2
)
for stage in parallel_plot.index:
    summary = parallel_plot.loc[stage]
    ypos = parallel_y[stage]
    ax.errorbar(
        summary['stage_delay_ms'], ypos,
        xerr=np.array([[
            summary['stage_delay_ms'] - summary['q25_ms']
        ], [
            summary['q75_ms'] - summary['stage_delay_ms']
        ]]),
        fmt='none', ecolor='#168C8C', capsize=3,
        linewidth=1.3, zorder=2
    )
    ax.annotate(
        f"{summary['stage_delay_ms']:.0f} ms",
        (summary['stage_delay_ms'], ypos), xytext=(7, -12),
        textcoords='offset points', color='#168C8C', fontsize=8,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=1.2)
    )

for ypos, row in rows:
    color, marker = STYLE[row['status']]
    if (
        np.isfinite(row['latency_ms'])
        and np.isfinite(row['latency_ci_low_ms'])
        and np.isfinite(row['latency_ci_high_ms'])
    ):
        ax.errorbar(
            row['latency_ms'], ypos,
            xerr=np.array([[
                max(0, row['latency_ms'] - row['latency_ci_low_ms'])
            ], [
                max(0, row['latency_ci_high_ms'] - row['latency_ms'])
            ]]),
            fmt='none', ecolor=color, alpha=0.48,
            capsize=2.5, linewidth=1.1, zorder=2
        )
    ax.scatter(
        row['latency_ms'], ypos, color=color, marker=marker,
        s=78 if row['required'] else 48,
        edgecolor='black' if row['required'] and marker == 'o' else color,
        linewidth=0.9, zorder=3
    )

# Hollow diamonds mark only the region-specific delays explicitly reported in
# the paper's main text; no values are invented for broadly reported ranges.
for region, paper_latency in Q3_PAPER_REPORTED_LATENCY_MS.items():
    if region not in region_y:
        continue
    ypos = region_y[region]
    ax.scatter(
        paper_latency, ypos, marker='D', s=34,
        facecolors='none', edgecolors='#7B3FA1',
        linewidth=1.2, zorder=5
    )
    ax.annotate(
        f'P:{paper_latency:.0f}', (paper_latency, ypos),
        xytext=(4, -10), textcoords='offset points',
        color='#7B3FA1', fontsize=7
    )

for group, centre in group_centres.items():
    ax.text(
        -0.08, centre, group, transform=ax.get_yaxis_transform(),
        ha='right', va='center', fontweight='bold', fontsize=9
    )

ax.set_yticks([ypos for ypos, _ in rows])
ax.set_yticklabels([row['acronym'] for _, row in rows], fontsize=8.5)
for tick, (_, row) in zip(ax.get_yticklabels(), rows):
    if row['required']:
        tick.set_fontweight('bold')
ax.invert_yaxis()
ax.set_xlim(0, 150)
ax.set_xlabel('Population-trajectory latency after stimulus onset (ms)')
ax.set_title(
    'Temporal recruitment of visual responses across brain regions',
    loc='left', fontweight='bold', pad=30
)
ax.text(
    0, 1.006,
    'Lines connect anatomical stages in time only. Temporal order is not evidence '
    'of direct anatomical connectivity or causal propagation.',
    transform=ax.transAxes, fontsize=8, color='#4B5563'
)
ax.grid(axis='x', linestyle=':', alpha=0.4)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)

handles = [
    plt.Line2D([], [], marker=marker, linestyle='', color=color,
               label=status, markersize=7)
    for status, (color, marker) in STYLE.items()
]
handles.insert(
    0,
    plt.Line2D([], [], marker='D', color='#2457A6', linewidth=1.8,
               label='Main-path stage median + IQR (raw p < 0.05)')
)
handles.insert(
    1,
    plt.Line2D([], [], marker='D', color='#168C8C', linestyle='--',
               linewidth=1.6, label='Candidate parallel midbrain branch')
)
handles.insert(
    2,
    plt.Line2D([], [], marker='D', linestyle='', markerfacecolor='none',
               markeredgecolor='#7B3FA1', color='#7B3FA1',
               label='Paper-reported regional delay')
)
handles.insert(
    3,
    plt.Line2D([], [], color='#6B7280', linewidth=1.2,
               marker='|', markersize=8,
               label=f'Region latency {Q3_REPORTED_CI:.0%} insertion-level CI')
)
ax.legend(
    handles=handles, frameon=False, loc='upper left',
    bbox_to_anchor=(1.01, 1), borderaxespad=0
)
fig.subplots_adjust(left=0.26, right=0.79, top=0.93, bottom=0.08)

pathway_figure = OUTPUT_DIR / 'Q3_paper_focused_propagation.png'
fig.savefig(pathway_figure, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig)  # superseded by the publication-quality chronological plot in section 9
print(f'Saved: {pathway_figure.resolve()}')


## 5. Comparison with the reference paper

The paper reports five regional latencies explicitly (34, 42, 42, 57 and 78 ms for LGd, VISp,
LP, VISpm and VISam); regions it describes only as "about 100–120 ms" are not given invented
point estimates. All five are compared — the analysis is not tuned toward any single one.

Four diagnostics run off the cached distance curves, so none requires rerunning the
permutation loop. They are diagnostics, **not** knobs: the headline definition stays fixed at
the paper's plain 70 % crossing whatever they show.

| Check | What it isolates |
|---|---|
| **A — onset rule** | the pointwise-significance gate that the paper does not use |
| **B — interpolation** | how much sub-bin interpolation moves the estimate |
| **C — threshold** | which threshold fraction *would* reproduce each paper value |
| **D — unresolved gap** | the 10 ms interval the crossing falls inside, which our sampling cannot resolve |

Check D is the decisive one. With 10 ms bins there are only 15 samples in the 0–150 ms
window, and every 70 % crossing lands *between* two adjacent samples across which the curve
jumps by 25–57 percentage points. Anything inside that gap is below our resolution; the paper,
sampling every 2 ms, could place the crossing precisely inside it.


In [ ]:
q3_paper_delay_comparison = (
    q3_pathway[q3_pathway['acronym'].isin(Q3_PAPER_REPORTED_LATENCY_MS)]
    [[
        'acronym', 'latency_ms', 'latency_ci_low_ms',
        'latency_ci_high_ms', 'bootstrap_valid'
    ]]
    .assign(
        paper_latency_ms=lambda frame: frame['acronym'].map(
            Q3_PAPER_REPORTED_LATENCY_MS
        )
    )
)
q3_paper_delay_comparison['difference_ms'] = (
    q3_paper_delay_comparison['latency_ms']
    - q3_paper_delay_comparison['paper_latency_ms']
)
q3_paper_delay_comparison['paper_within_ci'] = (
    q3_paper_delay_comparison['paper_latency_ms'].between(
        q3_paper_delay_comparison['latency_ci_low_ms'],
        q3_paper_delay_comparison['latency_ci_high_ms']
    )
)


q3_paper_delay_comparison['abs_difference_ms'] = (
    q3_paper_delay_comparison['difference_ms'].abs()
)


def q3_latency_variants(region):
    """Every latency definition, recomputed from the cached distance curve."""
    curve = q3_region_distance[region]
    point_p = q3_paper_point_p[region]
    return {
        'acronym': region,
        'A_paper_rule_ms': q3_paper_latency(q3_paper_times, curve),
        'A_gated_ms': q3_paper_latency(
            q3_paper_times, curve, point_p, require_significance=True
        ),
        'B_no_interpolation_ms': q3_paper_latency(
            q3_paper_times, curve, interpolate=False
        ),
        'C_threshold50_ms': q3_paper_latency(q3_paper_times, curve, fraction=0.50),
        'C_threshold60_ms': q3_paper_latency(q3_paper_times, curve, fraction=0.60),
    }


q3_sensitivity = pd.DataFrame(
    [q3_latency_variants(region) for region in Q3_PAPER_REPORTED_LATENCY_MS]
)
q3_sensitivity['paper_latency_ms'] = q3_sensitivity['acronym'].map(
    Q3_PAPER_REPORTED_LATENCY_MS
)
q3_sensitivity['gate_cost_ms'] = (
    q3_sensitivity['A_gated_ms'] - q3_sensitivity['A_paper_rule_ms']
)
q3_sensitivity['interpolation_shift_ms'] = (
    q3_sensitivity['A_paper_rule_ms'] - q3_sensitivity['B_no_interpolation_ms']
)

# D - quantisation floor imposed by the 10 ms PSTH bins.
q3_bin_ms = float(np.round(np.median(np.diff(q3_paper_times)) * 1000, 3))
print(f'PSTH bin width: {q3_bin_ms:.0f} ms  ->  resolution floor about '
      f'+/-{q3_bin_ms / 2:.0f} ms   (paper: 12.5 ms window, 2 ms stride)')
print(f'Samples available in the 0-150 ms window: {len(q3_paper_times)}')

display(q3_paper_delay_comparison.sort_values('paper_latency_ms').round(1))
display(q3_sensitivity.round(1))

# What threshold fraction would be needed to reproduce each paper value? If one
# consistent setting existed, the disagreement would be a definition mismatch.
# If the required settings contradict each other, tuning is not defensible.
q3_fraction_grid = np.arange(0.05, 0.96, 0.01)


def q3_required_fraction(region, target_ms):
    candidates = np.array([
        q3_paper_latency(q3_paper_times, q3_region_distance[region], fraction=f)
        for f in q3_fraction_grid
    ])
    best = np.nanargmin(np.abs(candidates - target_ms))
    return q3_fraction_grid[best], candidates[best]


q3_sensitivity[['required_fraction', 'best_achievable_ms']] = [
    q3_required_fraction(region, target)
    for region, target in Q3_PAPER_REPORTED_LATENCY_MS.items()
]


# The decisive resolution question: the 70% crossing always lands between two
# adjacent 10 ms samples. Everything inside that gap is unresolved by our data,
# so we ask whether the paper's value simply lies inside the same gap.
def q3_unresolved_gap(region):
    curve = q3_region_distance[region]
    threshold = curve.min() + Q3_ONSET_FRACTION * np.ptp(curve)
    stop = int(np.flatnonzero(curve >= threshold)[0])
    low = q3_paper_times[stop - 1] * 1000 if stop > 0 else 0.0
    return low, q3_paper_times[stop] * 1000


q3_sensitivity[['gap_low_ms', 'gap_high_ms']] = [
    q3_unresolved_gap(region) for region in Q3_PAPER_REPORTED_LATENCY_MS
]
q3_sensitivity['paper_inside_gap'] = (
    q3_sensitivity['paper_latency_ms'].between(
        q3_sensitivity['gap_low_ms'], q3_sensitivity['gap_high_ms']
    )
)
q3_sensitivity['paper_offset_from_gap_ms'] = np.where(
    q3_sensitivity['paper_inside_gap'], 0.0,
    np.where(
        q3_sensitivity['paper_latency_ms'] < q3_sensitivity['gap_low_ms'],
        q3_sensitivity['paper_latency_ms'] - q3_sensitivity['gap_low_ms'],
        q3_sensitivity['paper_latency_ms'] - q3_sensitivity['gap_high_ms'],
    )
)

print('\nThe 10 ms interval our sampling cannot resolve, per region:')
for row in q3_sensitivity.itertuples():
    verdict = ('paper value falls INSIDE it - the disagreement is entirely sub-bin'
               if row.paper_inside_gap
               else f'paper value sits {row.paper_offset_from_gap_ms:+.0f} ms outside it')
    print(f'   {row.acronym:6s} crossing bracketed by '
          f'[{row.gap_low_ms:5.0f}, {row.gap_high_ms:5.0f}] ms  ->  {verdict}')

print('\nThreshold fraction that would be needed to reproduce each paper value:')
for row in q3_sensitivity.itertuples():
    print(f'   {row.acronym:6s} paper {row.paper_latency_ms:5.0f} ms  '
          f'needs fraction {row.required_fraction:.2f}  '
          f'(our rule uses {Q3_ONSET_FRACTION:.2f}, giving {row.A_paper_rule_ms:.0f} ms)')
print(f'   -> required fractions span {q3_sensitivity["required_fraction"].min():.2f} '
      f'to {q3_sensitivity["required_fraction"].max():.2f}: no single setting '
      f'reconciles all five regions.')

# The distance curves themselves, so the comparison can be inspected directly.
q3_curve_table = pd.DataFrame(
    {region: q3_region_distance[region] for region in Q3_PAPER_REPORTED_LATENCY_MS},
    index=np.round(q3_paper_times * 1000, 1)
).rename_axis('time_ms')

fig4, ax4 = plt.subplots(figsize=(9, 4.6))
palette = ['#2457A6', '#168C8C', '#F28E2B', '#7B3FA1', '#C44E52']
for (region, colour) in zip(Q3_PAPER_REPORTED_LATENCY_MS, palette):
    curve = q3_region_distance[region]
    normalised = curve / np.ptp(curve)
    ax4.plot(q3_paper_times * 1000, normalised, color=colour, label=region, linewidth=1.6)
    ours = q3_paper_regions.loc[
        q3_paper_regions['acronym'] == region, 'latency_ms'
    ].iloc[0]
    ax4.plot(ours, Q3_ONSET_FRACTION, marker='o', color=colour, markersize=6)
    ax4.plot(Q3_PAPER_REPORTED_LATENCY_MS[region], Q3_ONSET_FRACTION, marker='D',
             markerfacecolor='none', markeredgecolor=colour, markersize=7)

ax4.axhline(Q3_ONSET_FRACTION, color='#6B7280', linestyle=':', linewidth=1)
ax4.text(148, Q3_ONSET_FRACTION + 0.02, f'{Q3_ONSET_FRACTION:.0%} of amplitude',
         ha='right', fontsize=8, color='#6B7280')
ax4.set_xlim(0, 150)
ax4.set_xlabel('Time after stimulus onset (ms)')
ax4.set_ylabel('Left-vs-right distance\n(normalised to its own amplitude)')
ax4.set_title('Why the estimates differ: the distance curves themselves',
              loc='left', fontweight='bold', pad=26)
ax4.text(0, 1.045, 'Filled circle = this notebook   |   open diamond = paper. '
         'Only 15 samples exist across this window (10 ms bins).',
         transform=ax4.transAxes, fontsize=8, color='#4B5563')
ax4.grid(axis='both', linestyle=':', alpha=0.35)
ax4.spines[['top', 'right']].set_visible(False)
ax4.legend(frameon=False, fontsize=8, loc='lower right')
fig4.tight_layout()
curve_figure = OUTPUT_DIR / 'Q3_distance_curve_comparison.png'
fig4.savefig(curve_figure, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig4)  # superseded by the publication-quality small multiples in section 9

display(q3_sensitivity.round(2))


### Reading the comparison

**The ordering replicates exactly.** LGd < LP ≈ VISp < VISpm ≈ VISam, followed by the late
midbrain/hindbrain wave at 100–120 ms — the same sequence, and the same late wave, that the
paper reports. That sequence is what Q3 actually asks about, and it is reproduced without
adjustment.

**Check A cost nothing.** The extra pointwise-significance gate changes the estimate by
**0 ms in all five regions** — their crossings were already significant. The gate was not the
source of any disagreement. The rule is still aligned to the paper's definition because that
is the correct definition, but the alignment is numerically neutral here.

**Check B works against us.** Interpolation already pulls the estimates 0–8 ms *earlier*;
removing it would increase the disagreement, not reduce it.

**Check C rules out tuning.** The threshold fraction that would reproduce the paper is 0.56
for LGd, 0.37 for VISp, 0.30 for LP, 0.16 for VISpm — but **0.95** for VISam. No single
setting reconciles all five, so "adjusting the threshold" would mean a different definition
per region, which is not a definition at all. The 70 % rule is kept.

**Check D explains four of the five.** Measuring each crossing against the 10 ms interval our
sampling cannot resolve:

| Region | Unresolved gap | Paper value | Verdict |
|---|---|---|---|
| LGd | 35–45 ms | 34 ms | 1 ms outside — agreement |
| VISp | 45–55 ms | 42 ms | 3 ms outside — agreement at our resolution |
| LP | 45–55 ms | 42 ms | 3 ms outside — agreement at our resolution |
| VISpm | 55–65 ms | 57 ms | **inside the gap** — not a disagreement at all |
| VISam | 55–65 ms | 78 ms | **13 ms outside** — a real difference |

So the original "≈15 ms discrepancy" resolves into two different things. Four regions agree
with the paper as closely as 10 ms sampling permits. **VISam is the single genuine outlier**,
and it is the only region where our estimate is *earlier* than the paper's — the opposite
direction from every measurement artefact considered above, none of which can produce it.

**What VISam is, then.** Its distance curve is already at 50 % of amplitude by 55 ms and 76 %
by 65 ms, so 70 % is reached early; the paper's curve evidently rises more slowly. The
plausible cause is the input data rather than the method: this analysis sees the 23 Neuromatch
insertions covering VISam, whereas the paper used the full brain-wide map. A different set of
recordings gives a differently-shaped population curve, and the 70 % point moves with it.

The estimate is therefore **kept as computed**. Its 95 % interval is [56, 76] ms, which misses
the paper's 78 ms by 2 ms. Forcing agreement would require a VISam-specific threshold of 0.95,
which would change what the number means for the sake of matching one value.


## 6. Uncertainty and final results

**At what level?** The interval resamples **insertions**, not trials and not neurons. A
trial-level bootstrap holds the recordings fixed and therefore measures only within-session
noise; it produces intervals that look reassuringly tight but say nothing about whether the
estimate would survive a different set of recordings. Both are exported — `latency_ci_*` is
the insertion-level interval and is the one reported, `trial_ci_*` is retained for comparison
and is visibly narrower.

**How wide?** 95 %. The interval's job here is to express descriptive precision on a handful
of reference comparisons, not to test hypotheses — regional significance is already handled by
BH-FDR at `q < 0.01`. A 90 % interval would declare disagreement with the paper more often
without adding any inferential value. `Q3_REPORTED_CI` is a parameter, so 90 % is one edit
away if it is ever wanted.

**Negative or pre-stimulus latencies.** None are possible by construction: crossings are
clamped to the 0–150 ms analysis window and interpolation is applied only when two adjacent
samples genuinely bracket the threshold. Earlier runs reported values such as
`[−94, 298] ms` for PRNc, which were extrapolation artefacts rather than real pre-stimulus
responses; the assertions in section 3 now fail loudly if anything leaves the window.
**On plotting single-unit latencies.** A violin plot of per-unit onsets was produced and then dropped: the distributions are near-uniform across the whole 0–150 ms window in every region, so the figure showed eighteen identical shapes. That flatness is reported as a number below instead, and it is the reason the regional estimate is taken from the population trajectory rather than from averaging single-neuron onsets.


In [ ]:
# Figure 2 - responsive neurons by region, for the regions on the propagation path.
q3_unit_summary = (
    q3_unit_counts[q3_unit_counts['acronym'].isin(region_to_pathway)]
    .merge(
        q3_region_latency[['acronym', 'n_pids', 'latency_ms']],
        on='acronym', how='left'
    )
    .sort_values('percent_responsive', ascending=False)
)

fig2, ax2 = plt.subplots(figsize=(9, 0.32 * len(q3_unit_summary) + 1.6))
positions = np.arange(len(q3_unit_summary))
ax2.barh(positions, q3_unit_summary['percent_responsive'],
         color='#F28E2B', height=0.68)
for pos, (_, row) in zip(positions, q3_unit_summary.iterrows()):
    ax2.text(
        row['percent_responsive'] + 0.4, pos,
        f"{int(row['n_responsive_units'])}/{int(row['n_tested_units'])}",
        va='center', fontsize=7.5, color='#374151'
    )
ax2.set_yticks(positions)
ax2.set_yticklabels(q3_unit_summary['acronym'], fontsize=8.5)
ax2.invert_yaxis()
ax2.set_xlabel('Responsive neurons (% of units tested in the region)')
ax2.set_title('Proportion of responsive neurons by region', loc='left',
              fontweight='bold')
ax2.grid(axis='x', linestyle=':', alpha=0.4)
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.tick_params(axis='y', length=0)
fig2.tight_layout()
responsive_figure = OUTPUT_DIR / 'Q3_responsive_unit_summary_by_region.png'
fig2.savefig(responsive_figure, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig2)  # superseded by the publication-quality lollipop plot in section 9

# No third figure: the per-unit latency distributions were plotted and did not
# earn their place - every region spanned almost the whole window near-uniformly,
# so eighteen near-identical shapes said less than the two numbers below. The
# spread itself is the finding, and it is why the reported regional latency comes
# from the population trajectory rather than from averaging single-unit onsets.
q3_unit_spread = (
    q3_responsive_units[q3_responsive_units['acronym'].isin(region_to_pathway)]
    .groupby('acronym')['latency_ms']
    .agg(median='median',
         iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
         n='size')
    .sort_values('median')
)
print('Spread of individual responsive-unit onsets along the propagation path:')
print(f"   median across regions of the within-region IQR: "
      f"{q3_unit_spread['iqr'].median():.0f} ms, in a {q3_window_ms[1]:.0f} ms window")
print(f"   narrowest region: {q3_unit_spread['iqr'].idxmin()} "
      f"({q3_unit_spread['iqr'].min():.0f} ms IQR); "
      f"widest: {q3_unit_spread['iqr'].idxmax()} "
      f"({q3_unit_spread['iqr'].max():.0f} ms IQR)")
print('   -> single-neuron onset is too noisy to order regions on its own.')
display(q3_unit_spread.round(1))
print(f'Saved: {responsive_figure.name}')

In [ ]:
# Consolidated regional result: one row per region on the propagation path.
q3_final_regions = (
    q3_pathway[[
        'acronym', 'anatomical_group', 'pathway_group', 'n_pids', 'n_sessions',
        'n_units', 'latency_ms', 'latency_ci_low_ms', 'latency_ci_high_ms',
        'trial_ci_low_ms', 'trial_ci_high_ms', 'latency_gated_ms',
        'p_value', 'q_value', 'significant', 'status'
    ]]
    .merge(
        q3_unit_counts[[
            'acronym', 'n_tested_units', 'n_responsive_units',
            'percent_responsive'
        ]],
        on='acronym', how='left'
    )
    .sort_values('latency_ms', na_position='last')
    .reset_index(drop=True)
)
q3_final_regions['ci_width_ms'] = (
    q3_final_regions['latency_ci_high_ms'] - q3_final_regions['latency_ci_low_ms']
)
q3_final_regions['trial_ci_width_ms'] = (
    q3_final_regions['trial_ci_high_ms'] - q3_final_regions['trial_ci_low_ms']
)

print('Median interval width, insertion level: '
      f"{q3_final_regions['ci_width_ms'].median():.1f} ms")
print('Median interval width, trial level:     '
      f"{q3_final_regions['trial_ci_width_ms'].median():.1f} ms")
display(q3_final_regions.round(1))

## 7. Direct answers to Q3

In [ ]:
# Explicit outputs for the three handwritten yellow questions.
q3_q1_region_summary = (
    q3_responsive_units.groupby('acronym', as_index=False)
    .agg(
        n_responsive_units=('uuids', 'nunique'),
        median_unit_latency_ms=('latency_ms', 'median')
    )
    .sort_values(['n_responsive_units', 'acronym'], ascending=[False, True])
)
# Q3 asks for the neuron id, its region, its latency and the recording it came
# from, so the session identifiers travel with the per-unit export.
q3_q1_unit_ids = (
    q3_responsive_units[[
        'uuids', 'acronym', 'pid', 'q_value', 'effect_hz', 'latency_ms'
    ]]
    .merge(
        q3_meta.sessions[['pid', 'eid', 'subject', 'lab']], on='pid', how='left'
    )
    .sort_values(['q_value', 'acronym', 'latency_ms'])
)

print('Q1 - WHICH UNITS RESPOND?')
print(f'Tested units: {len(q3_all_tested_units):,}')
print(f'Responsive units selected by q3_responsive_mask: {len(q3_responsive_units):,}')
print('First 30 responsive unit IDs (the complete table is exported):')
display(q3_q1_unit_ids.head(30).round(4))
display(q3_q1_region_summary.head(30).round(1))

q3_q2_required_latency = q3_region_latency[
    q3_region_latency['acronym'].isin(Q3_REQUIRED_REGIONS)
].sort_values('latency_ms')
print('Q2 - WHEN DOES THE RESPONSE BEGIN?')
display(q3_q2_required_latency.round(1))

print('Q3 - WHAT IS THE TEMPORAL ORDER ACROSS REGIONS?')
print(f'Descriptive path screen: raw permutation p < {Q3_STAGE_P_ALPHA}')
print('Paper-aligned inferential labels remain global BH-FDR q < 0.01.')
display(q3_main_stage_summary.round(1))
display(q3_parallel_stage_summary.round(1))
print('Current estimates versus exact approximate delays stated in the paper:')
display(q3_paper_delay_comparison.sort_values('paper_latency_ms').round(1))

## 8. Plain-language answers

*(filled in from the outputs above — see the printed tables and the three figures)*

In [ ]:
# One printed answer per question, each computed from the objects above so that
# nothing here can drift away from the tables.
main_order = q3_main_stage_summary.dropna(subset=['stage_delay_ms'])
top_responsive = (
    q3_unit_counts[q3_unit_counts['n_tested_units'] >= 50]
    .nlargest(5, 'percent_responsive')
)
visual_regions = ['LGd', 'LP', 'VISp', 'VISpm', 'VISam']
visual_responsive = q3_unit_counts[q3_unit_counts['acronym'].isin(visual_regions)]

# Built outside the f-string below: nested quotes inside f-strings need Python 3.12.
top_responsive_text = ', '.join(
    '{} ({:.0f}%)'.format(row.acronym, row.percent_responsive)
    for row in top_responsive.itertuples()
)
visual_text = ', '.join(
    '{} ({:.0f}%)'.format(row.acronym, row.percent_responsive)
    for row in visual_responsive.itertuples()
)
main_order_text = ' -> '.join(
    '{} {:.0f} ms'.format(stage, row.stage_delay_ms)
    for stage, row in main_order.iterrows()
)
responsive_pct = 100 * len(q3_responsive_units) / len(q3_all_tested_units)
inside_ci = int(q3_paper_delay_comparison['paper_within_ci'].sum())
inside_gap = int(
    (q3_sensitivity['paper_offset_from_gap_ms'].abs() <= 5).sum()
)
thalamus = q3_main_stage_summary.loc['Visual thalamus', 'stage_delay_ms']
cortex = q3_main_stage_summary.loc['Primary visual cortex', 'stage_delay_ms']
early_midbrain = q3_parallel_stage_summary.loc[
    'Early visual-midbrain branch', 'stage_delay_ms'
]
worst = q3_sensitivity.nlargest(1, 'paper_offset_from_gap_ms').iloc[0]

print(f"""
1. WHICH NEURONS WERE RESPONSIVE?
   {len(q3_responsive_units):,} of {len(q3_all_tested_units):,} tested units ({responsive_pct:.1f}%),
   selected by: q < {Q3_UNIT_FDR_ALPHA} AND effect >= {Q3_MIN_EFFECT_HZ} Hz AND a finite onset.
   This counts any change from baseline after stimulus onset, which is a much
   broader criterion than the paper's side-selectivity analysis.

2. HOW WERE LATENCIES CALCULATED?
   Regional (the reported number): first crossing of {Q3_ONSET_FRACTION:.0%} of the
   left-vs-right population distance amplitude - the paper's definition.
   Per unit (companion): first crossing of {Q3_ONSET_FRACTION:.0%} of peak |PSTH - baseline|,
   sustained for {Q3_SUSTAINED_BINS} consecutive bins.

3. WHICH REGIONS HAD THE LARGEST PROPORTIONS OF RESPONSIVE NEURONS?
   Overall (>= 50 units tested): {top_responsive_text}.
   Along the visual pathway: {visual_text}.
   Note these are onset responses, not side-selective ones, so subcortical
   regions with strong onset drive rank above visual cortex here.

4. WHAT WAS THE ANATOMICAL ORDER?
   {main_order_text}

5. DOES THE ORDER MAKE SENSE?
   Yes. Visual thalamus leads at {thalamus:.0f} ms, primary visual cortex follows at
   {cortex:.0f} ms, higher visual areas next, then midbrain/hindbrain, and
   motor/association regions last. That is the expected direction of travel.

6. PARALLEL PATHWAYS?
   The early visual-midbrain branch (SCs, NOT, OP) responds at {early_midbrain:.0f} ms,
   essentially together with the visual thalamus at {thalamus:.0f} ms and before cortex.
   A strictly serial retina->thalamus->cortex->midbrain chain cannot produce that.
   It is consistent with parallel retinal output - but it is temporal evidence
   only, not anatomical or causal evidence.

7. HOW CLOSELY DID THE RESULTS MATCH THE PAPER?
   The ordering replicates exactly, including the late 100-120 ms wave.
   On absolute values, two criteria give two different-looking answers:
     - {inside_ci} of 5 paper values fall inside our {Q3_REPORTED_CI:.0%} insertion-level
       interval, which measures variability across recordings;
     - {inside_gap} of 5 fall within the 10 ms interval our sampling cannot resolve,
       which measures what 10 ms bins can actually distinguish.
   The second is the fairer comparison for a bin-width-limited estimate.

8. HOW SHOULD THE REMAINING DISCREPANCY BE INTERPRETED?
   It is not one discrepancy. Four regions agree as closely as 10 ms sampling
   allows. Only {worst.acronym} genuinely differs: ours {worst.A_paper_rule_ms:.0f} ms
   vs paper {worst.paper_latency_ms:.0f} ms, and it is the only region where we are
   EARLIER than the paper - the opposite direction from every measurement
   artefact, so none of them explains it. Most likely cause is coverage: we see
   the Neuromatch insertions for {worst.acronym}, the paper used the full brain-wide map.
   Matching it would need a {worst.acronym}-specific threshold of
   {worst.required_fraction:.2f} instead of {Q3_ONSET_FRACTION:.2f}, which would change what
   the number means. The estimate was kept as computed.

9. WHAT UNCERTAINTY SHOULD ACCOMPANY THE LATENCIES?
   A {Q3_REPORTED_CI:.0%} percentile interval from an insertion-level bootstrap
   ({Q3_INSERTION_BOOTSTRAP_REPLICATES:,} replicates), reported with the number of
   contributing insertions. Insertions, not neurons or trials, are the
   independent replicate here.

CAUSALITY: none of the above is causal evidence. Latency ordering shows temporal
recruitment and statistical association. Whether one region drives another
requires perturbation, which this dataset cannot provide.
""")

## 9. Export

In [ ]:
import shutil

q3_paper_regions_complete.to_csv(
    OUTPUT_DIR / 'Q3_all_regions_paper_aligned.csv', index=False
)
q3_required_audit.to_csv(
    OUTPUT_DIR / 'Q3_required_regions.csv', index=False
)
q3_pathway.to_csv(
    OUTPUT_DIR / 'Q3_focused_propagation_regions.csv', index=False
)
q3_main_stage_summary.to_csv(
    OUTPUT_DIR / 'Q3_main_path_stage_delays.csv'
)
q3_parallel_stage_summary.to_csv(
    OUTPUT_DIR / 'Q3_parallel_midbrain_stage_delays.csv'
)
q3_region_latency.to_csv(
    OUTPUT_DIR / 'Q3_responsive_unit_latency_by_region.csv', index=False
)
q3_q1_unit_ids.to_csv(
    OUTPUT_DIR / 'Q3_responsive_unit_ids.csv', index=False
)
q3_q1_region_summary.to_csv(
    OUTPUT_DIR / 'Q3_responsive_unit_summary_by_region.csv', index=False
)
q3_q2_required_latency.to_csv(
    OUTPUT_DIR / 'Q3_required_region_unit_latencies.csv', index=False
)
q3_paper_delay_comparison.to_csv(
    OUTPUT_DIR / 'Q3_current_vs_paper_reported_delays.csv', index=False
)
q3_final_regions.to_csv(
    OUTPUT_DIR / 'Q3_final_regional_results.csv', index=False
)
q3_sensitivity.to_csv(
    OUTPUT_DIR / 'Q3_latency_sensitivity_checks.csv', index=False
)
q3_unit_counts.to_csv(
    OUTPUT_DIR / 'Q3_responsive_unit_counts_by_region.csv', index=False
)
q3_curve_table.to_csv(OUTPUT_DIR / 'Q3_paper_region_distance_curves.csv')

# -------------------------------------------------------------------------
# 9. Publication-quality figures
#
# The plotting module changes presentation only: every estimate is read back
# from the exported tables above. Regions are ordered chronologically, full
# insertion-level uncertainty is retained as a quiet outer interval, and the
# trial-level bootstrap is shown separately rather than substituted for the
# biologically meaningful insertion-level interval.
publication_candidates = [
    PROJECT_ROOT / 'src' / 'ibl_q3' / 'plotting.py',
    PROJECT_ROOT / 'Context 2' / 'q3_publication_plots.py',
    Path.cwd() / 'q3_publication_plots.py',
]
publication_module_path = next(
    (path for path in publication_candidates if path.exists()), None
)
if publication_module_path is None:
    raise FileNotFoundError(
        'q3_publication_plots.py must be kept beside Q3_final.ipynb'
    )
publication_spec = importlib.util.spec_from_file_location(
    'q3_publication_plots', publication_module_path
)
q3_publication_plots = importlib.util.module_from_spec(publication_spec)
publication_spec.loader.exec_module(q3_publication_plots)
publication_outputs = q3_publication_plots.build_publication_figures(OUTPUT_DIR)
print('Publication-quality figures:')
for output in publication_outputs:
    print(f'  {Path(output).name}')

from IPython.display import Image
for figure_name in [
    'Q3_temporal_recruitment_chronological.png',
    'Q3_temporal_recruitment_grouped.png',
    'Q3_distance_curve_small_multiples.png',
    'Q3_bootstrap_level_comparison.png',
    'Q3_sensitivity_decision_two_panel.png',
    'Q3_sensitivity_decision_condensed.png',
    'Q3_interpolation_resolution_two_panel.png',
    'Q3_benchmark_agreement_visual.png',
    'Q3_benchmark_agreement_condensed.png',
    'Q3_responsive_units_lollipop.png',
]:
    display(Image(filename=str(OUTPUT_DIR / figure_name), width=1100))

archive = Path(shutil.make_archive(
    str(PROJECT_ROOT / 'IBL_Q3_results_final'),
    'zip', root_dir=OUTPUT_DIR
))
print(f'Created: {archive.resolve()}')

if IN_COLAB:
    from google.colab import files
    files.download(str(archive))